# ML-05 — Feature Vector and Leakage/Privacy Check

Full-depth companion to the ML-04 March 2026 data-contract work. It uses the real warehouse partition and does not store credentials.


## 1. Build the feature vector

Decision moment: after day *t* has closed, before the next day. The label is tomorrow’s click indicator, so no tomorrow information belongs in `X`.


In [1]:
from pathlib import Path
import duckdb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

paths=list((Path.home()/'.cache'/'huggingface'/'hub').rglob('fact_content_daily_performance/month=2026-03/data_0.parquet'))
if not paths: raise RuntimeError('Download March with HF_TOKEN outside this notebook; do not paste tokens.')
con=duckdb.connect(); 
safe_path=str(paths[0]).replace(chr(39),chr(39)*2); 

con.execute(f"CREATE VIEW march AS SELECT * FROM read_parquet('{safe_path}')")
sql='''WITH p AS (SELECT report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,NULLIF(gsc_avg_position,0) pos,LEAD(gsc_clicks) OVER(PARTITION BY client_hash_id,content_hash_id ORDER BY report_date) nxt FROM march WHERE gsc_data_available IS TRUE) SELECT log(1+gsc_impressions) log_impressions_t,log(1+gsc_clicks) log_clicks_t,COALESCE(pos,100.0) gsc_avg_position_t,COALESCE(gsc_clicks::DOUBLE/NULLIF(gsc_impressions,0),0.0) ctr_t,dayofweek(report_date) day_of_week,(nxt>0)::INTEGER next_day_has_click FROM p WHERE report_date<DATE '2026-03-31' AND nxt IS NOT NULL USING SAMPLE 250000 ROWS (reservoir,202603)'''
frame=con.sql(sql).df(); 

features=['log_impressions_t','log_clicks_t','gsc_avg_position_t','ctr_t','day_of_week']; 
print('Feature vector shape:',frame.shape); 
display(frame.head())


Feature vector shape: (237638, 6)


,log_impressions_t,log_clicks_t,gsc_avg_position_t,ctr_t,day_of_week,next_day_has_click
0,1.431364,0.0,18.884615,0.0,5,0
1,1.041393,0.0,0.600000,0.0,4,0
2,0.845098,0.0,42.833333,0.0,5,0
3,1.278754,0.0,12.055556,0.0,6,1
4,1.544068,0.0,22.323529,0.0,5,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

- `log_impressions_t`: `log(1 + impressions)` controls skew; missing/zero becomes 0; numeric; available after today’s GSC export.
- `log_clicks_t`: `log(1 + clicks)` controls skew; missing/zero becomes 0; numeric; available after today’s GSC export.
- `gsc_avg_position_t`: current GSC position summary; source zero is treated as no position and filled with conservative 100; numeric; available after today’s GSC export.
- `ctr_t`: clicks / impressions, with zero denominator mapped to 0; numeric; available after today’s counts are known.
- `day_of_week`: 0–6 calendar encoding; no missingness; low-cardinality calendar categorical/ordinal input; known before the day begins.

Hashes stay outside the vector: they are context for grouping only, never model features.


In [2]:
print(frame[features].isna().sum().rename('missing_values').to_frame())
print('Label rate:',round(frame.next_day_has_click.mean(),4))


                    missing_values
log_impressions_t                0
log_clicks_t                     0
gsc_avg_position_t               0
ctr_t                            0
day_of_week                      0
Label rate: 0.1167


## 3. The leakage hunt

Attack: copy `next_day_has_click` into `leaked_next_day_has_click`. It is intentionally invalid because it is computed from the outcome being predicted. A near-perfect score is evidence of leakage, not a useful model.


In [3]:
cut=int(len(frame)*.8); 
train,test=frame.iloc[:cut].copy(),frame.iloc[cut:].copy()

honest=LogisticRegression(max_iter=200,random_state=202603).fit(train[features],train.next_day_has_click)
honest_auc=roc_auc_score(test.next_day_has_click,honest.predict_proba(test[features])[:,1])

leak='leaked_next_day_has_click'; 
train[leak]=train.next_day_has_click; 
test[leak]=test.next_day_has_click

bad=LogisticRegression(max_iter=200,random_state=202603).fit(train[features+[leak]],train.next_day_has_click)
bad_auc=roc_auc_score(test.next_day_has_click,bad.predict_proba(test[features+[leak]])[:,1])
print(f'Leaky AUC (invalid): {bad_auc:.4f}'); 
print(f'Honest AUC: {honest_auc:.4f}')

train.drop(columns=leak,inplace=True); 
test.drop(columns=leak,inplace=True); 
assert leak not in features and leak not in train.columns; 
print('Leak removed. Final feature list:',features)


Leaky AUC (invalid): 1.0000
Honest AUC: 0.8741
Leak removed. Final feature list: ['log_impressions_t', 'log_clicks_t', 'gsc_avg_position_t', 'ctr_t', 'day_of_week']


## 4. What I excluded and why

- `next_day_has_click`, next-day clicks, and all other *t+1* measures: outcome/future leakage.
- `client_hash_id` and `content_hash_id`: pseudonymous IDs can let a model memorize rather than generalize.
- `ga4_*`: GA4 is unavailable for part of the panel; using zero-filled values would confuse unavailable data with no engagement.
- June 2026 sample partition: final month is sealed and not used to develop label logic.

## Self-check

- Executable vector build
- Meaning, missingness, categorical handling, and availability listed
- Real leakage test shown then removed
- No names, URLs, queries, or tokens exposed
